In [84]:
import pandas as pd
import os

In [85]:
os.getcwd()

'c:\\Users\\olivi\\Downloads\\music_recommendor'

In [86]:
os.listdir()

['app.py', 'musicrecommendor.ipynb', 'spotifymillsongdata.xls']

In [87]:
df=pd.read_csv("spotifymillsongdata.xls")

In [88]:
df.head()

,artist,song,link,text
0,ABBA,Ahe's My Kind Of Girl,/a/abba/ahes+my+kind+of+girl_20598417.html,"Look at her face, it's a wonderful face \r\nA..."
1,ABBA,"Andante, Andante",/a/abba/andante+andante_20002708.html,"Take it easy with me, please \r\nTouch me gen..."
2,ABBA,As Good As New,/a/abba/as+good+as+new_20003033.html,I'll never know why I had to go \r\nWhy I had...
3,ABBA,Bang,/a/abba/bang_20598415.html,Making somebody happy is a question of give an...
4,ABBA,Bang-A-Boomerang,/a/abba/bang+a+boomerang_20002668.html,Making somebody happy is a question of give an...


In [89]:
df.isna().sum()

artist    0
song      0
link      0
text      0
dtype: int64

In [90]:
df.duplicated().sum()

np.int64(0)

In [91]:
#the no of rows in the dataset
df.shape[0]

57650

In [92]:
df = df.drop(columns=["link"]).reset_index(drop=True)

In [93]:
df.columns

Index(['artist', 'song', 'text'], dtype='object')

In [94]:
df["text"][0]

"Look at her face, it's a wonderful face  \r\nAnd it means something special to me  \r\nLook at the way that she smiles when she sees me  \r\nHow lucky can one fellow be?  \r\n  \r\nShe's just my kind of girl, she makes me feel fine  \r\nWho could ever believe that she could be mine?  \r\nShe's just my kind of girl, without her I'm blue  \r\nAnd if she ever leaves me what could I do, what could I do?  \r\n  \r\nAnd when we go for a walk in the park  \r\nAnd she holds me and squeezes my hand  \r\nWe'll go on walking for hours and talking  \r\nAbout all the things that we plan  \r\n  \r\nShe's just my kind of girl, she makes me feel fine  \r\nWho could ever believe that she could be mine?  \r\nShe's just my kind of girl, without her I'm blue  \r\nAnd if she ever leaves me what could I do, what could I do?\r\n\r\n"

In [95]:
df=df.sample(57650)

 DATA CLEANING /TEXT FORMATTING

In [96]:
df["text"]= df["text"].str.lower().replace(r'^\w\s','').replace(r'\n','',regex=True)


In [97]:
import nltk
from nltk.stem.porter import PorterStemmer

stemmer = PorterStemmer()

def token(txt):
    token = nltk.word_tokenize(txt)
    a = [stemmer.stem(w) for w in token]
    return " ".join(a)

RECOMMENDOR SYSTEM

In [106]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidf = TfidfVectorizer(max_features=30000, stop_words="english")
matrix = tfidf.fit_transform(df["text"])

def recommender(song_name):
    matches = df[df["song"] == song_name]

    if matches.empty:
        return []

    index = matches.index[0]

    # Compute similarity for just one song
    distances = cosine_similarity(matrix[index], matrix).flatten()

    song_indices = distances.argsort()[::-1][1:6]

    return df.iloc[song_indices]["song"].tolist()


In [107]:

df[df["song"].str.contains("As Good As New", case=False, na=False)]


,artist,song,text
2,ABBA,As Good As New,i'll never know why i had to go \rwhy i had t...


In [108]:
df.head()

,artist,song,text
14242,Nirvana,Sifting,afraid to grade \rwouldn't it be fun \rcross...
17520,Reo Speedwagon,I Do' Wanna Know,you have said as much as you can say \ryour h...
10198,Keith Green,To Obey Is Better Than Sacrifice,to obey is better than sacrifice \ri don't ne...
25457,Bing Crosby,Avalon,ev'ry morning mem'ries stray \racross the sea...
10940,Kris Kristofferson,Crossing The Border,"hey, take it and run, child \ri'll ask you no..."


In [109]:
df[df["song"]=="In The Garden"].index[0]

np.int64(25875)

In [110]:
df[df["song"].str.contains("Truly", case=False, na=False)]

,artist,song,text
11787,Lionel Richie,Truly,"girl, tell me only this \rthat i'll have your..."
51384,Savage Garden,Truly Madly Deeply,"i'll be your dream, i'll be your wish, i'll be..."
31784,Erasure,"Truly, Madly, Deeply",however hard they try to hold you back \rso y...
19187,The Temptations,"I Truly, Truly Believe",i'm sending you red roses and violets. \r(vio...
30947,Electric Light Orchestra,Yours Truly 2095,"i sent a message to another time, \rbut as th..."


In [105]:
import pickle

pickle.dump(df, open("df.pkl", "wb"))
pickle.dump(tfidf, open("tfidf.pkl", "wb"))
pickle.dump(matrix, open("matrix.pkl", "wb"))